# StatsPAI 与 Stata 并排实跑

### 同一份数据，两边现场算，自动比对

`statspai_vs_stata_5min.ipynb` 里的 Stata 输出是事先跑好贴进去的。这一份更进一步，在**同一个 notebook 里现场调用你电脑上的 Stata**，把它算出来的系数和标准误直接取回 Python，和 StatsPAI 的结果逐个相减。两边的数字都不是手敲的。

**运行这份 notebook 需要本机装有 Stata 17 或更新的版本**（用的是 Stata 自带的 `pystata`），以及外部命令 `reghdfe`、`csdid`、`drdid`、`rdrobust`。没有 Stata 也可以直接往下读，文件里保存了在 Stata 18 MP 上执行的全部输出。想要一份不依赖 Stata 的版本，请看 `statspai_vs_stata_5min.ipynb`。

内核是 **Python**。以 `%%stata` 开头的单元格里写的是原汁原味的 Stata 代码，由 Stata 执行，输出也是 Stata 的。

| 节 | Stata 命令 | StatsPAI 函数 |
| --- | --- | --- |
| 0 | 连接 Stata，把数据送过去 | |
| 1 | `regress ..., vce(robust)` | `sp.regress` |
| 2 | `reghdfe` | `sp.hdfe_ols` |
| 3 | `ivregress 2sls` | `sp.ivreg` |
| 4 | `csdid` + `estat simple` | `sp.callaway_santanna` + `sp.aggte` |
| 5 | `rdrobust` | `sp.rdrobust` |
| 6 | 一段 do 文件 | `sp.stata` |
| 7 | 汇总，逐项相减 | |

## 0. 连接 Stata

`pystata` 随 Stata 一起安装，在 Stata 安装目录的 `utilities` 文件夹里。下面这个单元格按操作系统猜一个默认路径。如果你的 Stata 装在别处，或者版本不是 MP，设置环境变量 `STATA_HOME` 和 `STATA_EDITION`（取 `mp`、`se` 或 `be`），或者直接改这两行。

In [1]:
import os
import platform
import sys
import warnings

warnings.filterwarnings("ignore")  # 教程里隐藏无关提示，正式分析时建议保留

default_home = {
    "Darwin": "/Applications/Stata",
    "Windows": r"C:\Program Files\Stata18",
    "Linux": "/usr/local/stata18",
}.get(platform.system(), "")
STATA_HOME = os.environ.get("STATA_HOME", default_home)
STATA_EDITION = os.environ.get("STATA_EDITION", "mp")

sys.path.append(os.path.join(STATA_HOME, "utilities"))
from pystata import config

config.init(STATA_EDITION, splash=False)   # 启动 Stata，并注册 %%stata 单元格魔法
from pystata import stata

import numpy as np
import pandas as pd
import statspai as sp

print("StatsPAI 版本:", sp.__version__)

StatsPAI 版本: 1.39.3


In [2]:
%%stata
set linesize 120
display "Stata " c(stata_version) " " c(edition_real)
foreach c in reghdfe csdid drdid rdrobust {
    capture which `c'
    display "`c': " cond(_rc == 0, "installed", "MISSING -- run: ssc install `c'")
}


. set linesize 120

. display "Stata " c(stata_version) " " c(edition_real)
Stata 18 MP

. foreach c in reghdfe csdid drdid rdrobust {
  2.     capture which `c'
  3.     display "`c': " cond(_rc == 0, "installed", "MISSING -- run: ssc install `c'")
  4. }
reghdfe: installed
csdid: installed
drdid: installed
rdrobust: installed

. 


三份数据都随 StatsPAI 自带。先在 Python 里读进来，后面每一节用 `%%stata -d 数据框 -force` 把同一个 DataFrame 送进 Stata，保证两边用的是同一份数据。

再写一个小工具，把每一节两边的数字记下来，最后统一比较。

In [3]:
card = sp.datasets.card_1995()                                   # Card (1995) 教育回报率
mpdta = sp.datasets.mpdta()                                      # 最低工资与青少年就业，县 × 年面板
senate = sp.datasets.lee_2008_senate().dropna().reset_index(drop=True)   # 参议院选举，断点回归

records = []


def record(method, quantity, stata_value, statspai_value):
    """记下一对数字，并当场打印。"""
    stata_value, statspai_value = float(stata_value), float(statspai_value)
    records.append((method, quantity, stata_value, statspai_value))
    print(f"{quantity:<12} Stata {stata_value: .10g}   StatsPAI {statspai_value: .10g}   差 {abs(stata_value - statspai_value):.1e}")

取回 Stata 的结果很直接。`%%stata -eret 名字` 把这个单元格运行之后的 `e()` 整个存成一个 Python 字典（`-ret` 对应 `r()`），矩阵是 NumPy 数组。`e(b)` 是一行系数，`e(V)` 是协方差矩阵，列的顺序就是 Stata 结果表里从上到下的顺序。

## 1. OLS + 稳健标准误

In [4]:
%%stata -d card -force -eret e_ols
regress lwage educ exper expersq black south smsa, vce(robust)


Linear regression                               Number of obs     =      3,010
                                                F(6, 3003)        =     217.74
                                                Prob > F          =     0.0000
                                                R-squared         =     0.2905
                                                Root MSE          =     .37419



------------------------------------------------------------------------------


             |               Robust
       lwage | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        educ |    .074009    .003642    20.32   0.000     .0668679    .0811501
       exper |   .0835958   .0067326    12.42   0.000     .0703948    .0967969
     expersq |  -.0022409   .0003181    -7.04   0.000    -.0028646   -.0016171
       black |  -.1896315   .0174324   -10.88   0.000    -.2238123   -.1554508
       south |  -.1248615   .0153508    -8.13   0.000    -.1549606   -.0947625
        smsa |    .161423   .0151751    10.64   0.000     .1316683    .1911776
       _cons |   4.733664   .0701577    67.47   0.000     4.596102    4.871226
------------------------------------------------------------------------------


In [5]:
ols = sp.regress(
    "lwage ~ educ + exper + expersq + black + south + smsa",
    data=card,
    robust="hc1",  # 等价于 Stata 的 vce(robust)
)

# educ 是 Stata 结果表的第一行，所以取 e(b) 的第 0 个、e(V) 的第 (0, 0) 个
record("OLS", "educ 系数", e_ols["e(b)"][0, 0], ols.params["educ"])
record("OLS", "educ 标准误", np.sqrt(e_ols["e(V)"][0, 0]), ols.std_errors["educ"])

educ 系数      Stata  0.0740089942   StatsPAI  0.0740089942   差 2.6e-15
educ 标准误     Stata  0.003642033531   StatsPAI  0.003642033531   差 2.0e-16


## 2. 高维固定效应 + 聚类标准误

In [6]:
%%stata -d mpdta -force -eret e_fe
reghdfe lemp treat, absorb(countyreal year) vce(cluster countyreal)

(MWFE estimator converged in 2 iterations)

HDFE Linear regression                            Number of obs   =      2,500
Absorbing 2 HDFE groups                           F(   1,    499) =      42.05
Statistics robust to heteroskedasticity           Prob > F        =     0.0000
                                                  R-squared       =     0.8294
                                                  Adj R-squared   =     0.7863
                                                  Within R-sq.    =     0.0205
Number of clusters (countyreal) =        500      Root MSE        =     0.0795

                           (Std. err. adjusted for 500 clusters in countyreal)
------------------------------------------------------------------------------
             |               Robust
        lemp | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
       treat |  -.0375051   .0057836    -6.48   0.

In [7]:
fe = sp.hdfe_ols("lemp ~ treat | countyreal + year", data=mpdta, cluster="countyreal")

record("固定效应", "treat 系数", e_fe["e(b)"][0, 0], fe.params["treat"])
record("固定效应", "treat 标准误", np.sqrt(e_fe["e(V)"][0, 0]), fe.std_errors["treat"])

treat 系数     Stata -0.03750507659   StatsPAI -0.03750507659   差 4.8e-16
treat 标准误    Stata  0.005783569517   StatsPAI  0.005783569517   差 1.7e-18


## 3. 工具变量（2SLS）

Stata 的 `small` 选项做小样本自由度修正并使用 t 分布，StatsPAI 的 `sp.ivreg` 默认就是这个口径。

In [8]:
%%stata -d card -force -eret e_iv
ivregress 2sls lwage exper expersq black south smsa (educ = nearc4), vce(robust) small


Instrumental variables 2SLS regression            Number of obs   =      3,010
                                                  F(  6,  3003)   =     131.70
                                                  Prob > F        =     0.0000
                                                  R-squared       =     0.2252
                                                  Adj R-squared   =     0.2237
                                                  Root MSE        =     .39103

------------------------------------------------------------------------------
             |               Robust
       lwage | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        educ |   .1322888   .0485779     2.72   0.007     .0370396    .2275381
       exper |    .107498   .0211375     5.09   0.000     .0660525    .1489434
     expersq |  -.0022841   .0003467    -6.59   0.000    -.0029639   -.0016042
       black |

In [9]:
iv = sp.ivreg(
    "lwage ~ (educ ~ nearc4) + exper + expersq + black + south + smsa",
    data=card,
    robust="hc1",
)

# ivregress 把内生变量排在结果表的第一行
record("2SLS", "educ 系数", e_iv["e(b)"][0, 0], iv.params["educ"])
record("2SLS", "educ 标准误", np.sqrt(e_iv["e(V)"][0, 0]), iv.std_errors["educ"])

educ 系数      Stata  0.13228884   StatsPAI  0.13228884   差 2.0e-14
educ 标准误     Stata  0.0485778603   StatsPAI  0.0485778603   差 4.7e-12


## 4. 交错 DID（Callaway-Sant'Anna）

`estat simple` 把各个组别-时期的 ATT 汇总成一个数，结果放在 `r()` 里，用 `-ret` 取回。

In [10]:
%%stata -d mpdta -force -ret r_cs
csdid lemp, ivar(countyreal) time(year) gvar(first_treat) method(dripw)
estat simple


. csdid lemp, ivar(countyreal) time(year) gvar(first_treat) method(dripw)


.

..

.

.

...

.

.

.

..


Difference-in-difference with Multiple Time Periods

                                                         Number of obs = 2,500
Outcome model  : least squares
Treatment model: inverse probability
------------------------------------------------------------------------------
             | Coefficient  Std. err.      z    P>|z|     [95% conf. interval]
-------------+----------------------------------------------------------------
g2004        |
 t_2003_2004 |  -.0472597   .0141767    -3.33   0.001    -.0750454   -.0194739
 t_2003_2005 |  -.0471138    .014997    -3.14   0.002    -.0765074   -.0177201
 t_2003_2006 |  -.0455279   .0147806    -3.08   0.002    -.0744973   -.0165585
 t_2003_2007 |  -.0277338   .0143663    -1.93   0.054    -.0558913    .0004236
-------------+----------------------------------------------------------------
g2006        |
 t_2003_2004 |  -.0058875   .0144345    -0.41   0.683    -.0341785    .0224035
 t_2004_2005 |  -.0172483   .0151936    -1.14   0.256    -

In [11]:
cs = sp.callaway_santanna(mpdta, y="lemp", g="first_treat", t="year", i="countyreal")
att = sp.aggte(cs, type="simple")

record("交错 DID", "总体 ATT", r_cs["r(b)"][0, 0], att.estimate)
record("交错 DID", "ATT 标准误", np.sqrt(r_cs["r(V)"][0, 0]), att.se)

总体 ATT       Stata -0.03297651381   StatsPAI -0.03297651381   差 6.9e-18
ATT 标准误      Stata  0.007764542536   StatsPAI  0.007764542536   差 1.7e-18


## 5. 断点回归

`rdrobust` 把稳健偏差校正的估计和标准误存在 `e(tau_bc)` 与 `e(se_tau_rb)` 里。

In [12]:
%%stata -d senate -force -eret e_rd
rdrobust y x, c(0) all


Sharp RD estimates using local polynomial regression.

      Cutoff c = 0 | Left of c  Right of c            Number of obs =       1297
-------------------+----------------------            BW type       =      mserd
     Number of obs |       595         702            Kernel        = Triangular
Eff. Number of obs |       360         323            VCE method    =         NN
    Order est. (p) |         1           1
    Order bias (q) |         2           2
       BW est. (h) |    17.754      17.754
       BW bias (b) |    28.028      28.028
         rho (h/b) |     0.633       0.633

Outcome: y. Running variable: x.
--------------------------------------------------------------------------------
            Method |   Coef.    Std. Err.    z     P>|z|    [95% Conf. Interval]
-------------------+------------------------------------------------------------
      Conventional |  7.4141     1.4587   5.0826   0.000     4.5551      10.2732
    Bias-corrected |  7.5065     1.4587   5.146

In [13]:
rd = sp.rdrobust(senate, y="y", x="x", c=0)

record("断点回归", "稳健估计", e_rd["e(tau_bc)"], rd.estimate)
record("断点回归", "稳健标准误", e_rd["e(se_tau_rb)"], rd.se)
record("断点回归", "带宽 h", e_rd["e(h_l)"], rd.model_info["bandwidth_h"])

稳健估计         Stata  7.506502483   StatsPAI  7.506502484   差 7.1e-10
稳健标准误        Stata  1.741258414   StatsPAI  1.741258414   差 2.6e-10
带宽 h         Stata  17.7543973   StatsPAI  17.7543973   差 6.5e-09


## 6. 一段 do 文件

同一段 Stata 代码，先交给 Stata，再原样交给 `sp.stata`。

In [14]:
do_file = """
gen exper2 = exper^2
regress lwage educ exper exper2 black south smsa, vce(robust)
test exper exper2
"""

In [15]:
%%stata -d card -force -ret r_test
gen exper2 = exper^2
regress lwage educ exper exper2 black south smsa, vce(robust)
test exper exper2


. gen exper2 = exper^2

. regress lwage educ exper exper2 black south smsa, vce(robust)

Linear regression                               Number of obs     =      3,010
                                                F(6, 3003)        =     217.74
                                                Prob > F          =     0.0000
                                                R-squared         =     0.2905
                                                Root MSE          =     .37419



------------------------------------------------------------------------------
             |               Robust
       lwage | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
        educ |    .074009    .003642    20.32   0.000     .0668679    .0811501
       exper |   .0835958   .0067326    12.42   0.000     .0703948    .0967969
      exper2 |  -.0022409   .0003181    -7.04   0.000    -.0028646   -.0016171
       black |  -.1896315   .0174324   -10.88   0.000    -.2238123   -.1554508
       south |  -.1248615   .0153508    -8.13   0.000    -.1549606   -.0947625
        smsa |    .161423   .0151751    10.64   0.000     .1316683    .1911776
       _cons |   4.733664   .0701577    67.47   0.000     4.596102    4.871226
------------------------------------------------------------------------------

. test exper exper2

 ( 1)  exper = 0
 ( 2)  exper2 = 0

       F(  2,  3003) =  178.29
      

In [16]:
out = sp.stata(do_file, data=card)   # 逐行翻译并运行，返回最后一条命令（test）的结果

record("do 文件", "联合检验 F", r_test["r(F)"], out["statistic"])
record("do 文件", "p 值", r_test["r(p)"], out["pvalue"])

联合检验 F       Stata  178.2864523   StatsPAI  178.2864523   差 5.5e-11
p 值          Stata  6.820249462e-74   StatsPAI  6.820249462e-74   差 3.3e-84


## 7. 汇总

In [17]:
compare = pd.DataFrame(records, columns=["方法", "统计量", "Stata", "StatsPAI"])
compare["绝对差"] = (compare["Stata"] - compare["StatsPAI"]).abs()
compare["相对差"] = compare["绝对差"] / compare["Stata"].abs().clip(lower=1e-300)

print(f"共 {len(compare)} 个数字，最大相对差 {compare['相对差'].max():.1e}")
compare.style.format({"Stata": "{:.10g}", "StatsPAI": "{:.10g}", "绝对差": "{:.1e}", "相对差": "{:.1e}"})

共 13 个数字，最大相对差 3.7e-10


,方法,统计量,Stata,StatsPAI,绝对差,相对差
0,OLS,educ 系数,0.0740089942,0.0740089942,2.6e-15,3.5e-14
1,OLS,educ 标准误,0.003642033531,0.003642033531,2.0e-16,5.6e-14
2,固定效应,treat 系数,-0.03750507659,-0.03750507659,4.8e-16,1.3e-14
3,固定效应,treat 标准误,0.005783569517,0.005783569517,1.7e-18,3.0e-16
4,2SLS,educ 系数,0.13228884,0.13228884,2.0e-14,1.5e-13
5,2SLS,educ 标准误,0.0485778603,0.0485778603,4.7e-12,9.6e-11
6,交错 DID,总体 ATT,-0.03297651381,-0.03297651381,6.9e-18,2.1e-16
7,交错 DID,ATT 标准误,0.007764542536,0.007764542536,1.7e-18,2.2e-16
8,断点回归,稳健估计,7.506502483,7.506502484,7.1e-10,9.4e-11
9,断点回归,稳健标准误,1.741258414,1.741258414,2.6e-10,1.5e-10


In [18]:
# 把比对写成断言。以后 StatsPAI 或 Stata 任何一边的结果变了，这个单元格会报错
assert compare["相对差"].max() < 1e-6, compare.sort_values("相对差").tail(3)
print("全部数字在 1e-6 的相对误差以内。")

全部数字在 1e-6 的相对误差以内。


怎么读这张表。

- 两列数字都是这次运行现场算出来的，Stata 那一列取自 Stata 内存里的 `e()` 和 `r()`，不经过屏幕显示的四舍五入。
- 相对差的上限 1e-6 是 StatsPAI 对「与 Stata 对齐」的定义。闭式估计量（OLS、固定效应、2SLS）的差距在机器精度附近，带宽选择和迭代求解的估计量（断点回归、DID）差距稍大，仍远在这条线以内。
- 这里只对了六组设定。StatsPAI 完整的跨语言对齐证据见 `docs/parity.md`，覆盖八十多个模块、R 和 Stata 两侧。

想把这套做法用在自己的研究上，流程是一样的。`%%stata -d 你的数据框 -force -eret 名字` 跑 Stata，旁边跑 StatsPAI，再把两个数相减。